In [ ]:
import numpy as np
from sklearn.metrics import root_mean_squared_error as rmse
from tqdm import tqdm, trange
from lib.trainer import ResShiftTrainer

In [ ]:
hr_size = 512
factor = 8
lr_size = hr_size // factor

var_name1 = f'resshift_8x_groklst_v3'
trainer_resshift = ResShiftTrainer(config_base_path = 'configs/base.yaml', config_var_path = 'configs/var.yaml', var_name = var_name1)
trainer_resshift.reload_model(model_path=f'models/{var_name1}_best.pth')
trainer_resshift.set_seed_during_diffusion(0)
trainer_resshift.load_groklst_dataset(zoom=factor)
testloader = trainer_resshift.testloader_groklst

lst_true_all, lst_pgdm_all = trainer_resshift.evaluate_after_train(testloader)
num_imgs = len(lst_true_all)

100%|██████████| 97/97 [01:07<00:00,  1.45it/s]


In [11]:
rmse_prd = np.mean(rmse(lst_true_all.reshape(num_imgs, -1).swapaxes(0, 1), 
             lst_pgdm_all.reshape(num_imgs, -1).swapaxes(0, 1), 
             multioutput='raw_values')) # mean of sample-wise RMSE
mae_prd = np.mean(np.abs(lst_true_all - lst_pgdm_all))
bias_prd = np.mean(lst_pgdm_all - lst_true_all)

cc_all = np.zeros(num_imgs)
for i in range(num_imgs):
    cc_all[i] = np.corrcoef(lst_true_all[i].flatten(), lst_pgdm_all[i].flatten())[0, 1]
cc_prd = np.mean(cc_all)

print(f'RMSE={rmse_prd:.4f} K, MAE={mae_prd:.4f} K, Bias={bias_prd:.4f} K, CC={cc_prd:.4f}')

RMSE=0.7491 K, MAE=0.5226 K, Bias=-0.0092 K, CC=0.9607
